# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which describes the structure and record sets (tables) in the dataset.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant descriptor URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Entities in the Croissant dataset are referenced by their `@id` property. We print available record set IDs, and for each set, list fields and columns. This is crucial for further referencing when loading or analyzing the data.

In [ ]:
# List all record set @id values in the dataset
record_sets = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
print('Available record sets:')
for rs in record_sets:
    print(' -', rs)

# Display fields for each record set (by @id)
for rs in metadata.to_json().get('recordSet', []):
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('field', [])
    if not fields:
        print('  No fields listed.')
        continue
    print('  Fields:')
    for f in fields:
        # f can be dict or @id string
        if isinstance(f, dict):
            print(f"   - {f.get('@id', f)}")
        else:
            print(f"   - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we demonstrate how to examine a record set by extracting its records into a DataFrame. Update the `record_set_ids` list with the relevant IDs for further processing.

In [ ]:
## Example: Load all available record sets into DataFrames
record_set_ids = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        if len(df.columns) > 0:
            print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No columns extracted for {record_set_id}.")
    except Exception as e:
        print(f"Could not convert records for {record_set_id} to DataFrame: {e}")

# For demonstration, choose the first available record set with data
active_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        active_record_set_id = k
        break
if active_record_set_id:
    print(f"Active record set selected for analysis: {active_record_set_id}")
    print(dataframes[active_record_set_id].head())
else:
    print('No dataframes with data were found!')

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records by numeric field, normalize values, and group by a key attribute.

> Update the `numeric_field_id` and `group_field_id` variables with the appropriate field `@id` values listed in the Data Overview section above. These are used for demonstration below.

In [ ]:
import numpy as np
from pandas.api.types import is_numeric_dtype

# Choose a numeric field from the columns in active_record_set_id
df = dataframes.get(active_record_set_id, pd.DataFrame())
if df.empty:
    print('No data loaded for EDA.')
else:
    # Try to automatically select a numeric column
    numeric_columns = [c for c in df.columns if is_numeric_dtype(df[c])]
    if not numeric_columns:
        print('No numeric fields detected for EDA.')
    else:
        # Use the first numeric column for demonstration
        numeric_field_id = numeric_columns[0]
        print(f'Using numeric field (@id): {numeric_field_id}')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in '{active_record_set_id}' where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try to guess a group field from category-like columns
        candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field_id]
        group_field_id = candidate_group_fields[0] if candidate_group_fields else None
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print('No suitable group field (non-numeric) found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The example below shows a histogram of the chosen numeric field. You can adjust `numeric_field_id` as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric data available for visualization.')

## 6. Conclusion

We have demonstrated how to:
- Access and parse FAIR^2 dataset metadata with the `mlcroissant` library
- List record sets and fields by their `@id`
- Load and manipulate one or more record sets as pandas DataFrames
- Perform simple EDA and visualization referencing dataset fields *by their `@id`*

All steps can be extended to custom record sets and fields by adjusting the `@id` variables as listed in your dataset's Croissant schema overview. For further processing, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant).